# 단일 클래스 알약 탐지기 학습

이 노트북은 단일 `pill` 클래스 탐지기의 데이터 로드, 학습, checkpoint 재개, validation 평가와 결과 저장을 수행한다. 모델, epoch loop, 지표와 checkpoint 로직은 `detector/` 모듈을 사용한다.

## 실행 환경

- **Colab:** GPU 런타임을 선택하고 저장소와 탐지 데이터 경로를 준비한 뒤 위에서부터 실행한다. 결과를 유지하려면 설정 셀의 `output_dir`을 Google Drive 경로로 변경한다.
- **로컬:** 저장소 루트에서 `uv sync --locked`를 실행하고 데이터 factory가 읽을 학습 패키지를 준비한다. Apple Silicon에서는 MPS를 사용할 수 있다.

## 학습 실행 설정

설정 셀의 `run_name`은 학습 실행을 구분하는 이름이며, 결과는 `<output_dir>/<run_name>/`에 저장된다. 작은 `max_epochs`로 전체 흐름을 먼저 확인하고 본 학습에서는 날짜·모델·데이터·목적·순번을 포함한 실행 이름을 사용한다.

같은 `run_name`의 결과가 있으면 덮어쓰기를 방지하기 위해 새 학습이 중단된다. 기존 학습을 이어가려면 같은 `run_name`을 사용하고 `config.resume_from`에 해당 실행의 `last.pt` 경로를 지정한다.

주요 결과물:

- `best.pt`: validation 성능이 가장 좋은 checkpoint
- `last.pt`: 마지막 epoch의 checkpoint
- `metrics.csv`: epoch별 학습 및 validation 지표
- `resolved_config.json`: 실제 학습에 사용된 설정
- validation 예측, 오류와 시각화 artifact

In [ ]:
# ruff: noqa: E402
import sys
from dataclasses import replace
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd

from configs.detector import load_detector_config
from detector.evaluate import evaluate_model
from detector.inference import (
    load_detector_inference_model,
    plot_detection_comparison,
    predict_detector,
    save_inference_artifacts,
)
from detector.train import train_detector
from detector.training_data import prepare_detector_data
from pipeline.pill_recognition import build_classifier_batch

In [ ]:
run_name = "20261104-fasterrcnn-r50-fpn-v2-det-default-baseline-02"
model_name = "fasterrcnn_resnet50_fpn_v2"
device = "mps"
show_progress = True
# tags = [
#     "task:detector",
#     "stage:baseline",
#     "model:fasterrcnn-r50-fpn-v2",
#     "data:det-default",
#     "change:baseline",
#     "device:mps",
# ]

output_dir = project_root / "outputs" / "detector"
data_dir = project_root / "data" / "detector"

batch_size = 1
max_epochs = 20
learning_rate = 0.0025
pretrained_backbone = True
image_min_size = 384
image_max_size = 640
resume_from = None

# 이미지와 bbox에 함께 적용되는 학습 증강 확률과 강도
horizontal_flip = 0.0
vertical_flip = 0.0
brightness = 0.0
contrast = 0.0

base_config = load_detector_config(project_root / "configs" / "detector.yaml")
config = replace(
    base_config,
    run_name=run_name,
    model_name=model_name,
    output_dir=output_dir,
    data_dir=data_dir,
    pretrained_backbone=pretrained_backbone,
    image_min_size=image_min_size,
    image_max_size=image_max_size,
    device=device,
    batch_size=batch_size,
    max_epochs=max_epochs,
    learning_rate=learning_rate,
    horizontal_flip=horizontal_flip,
    vertical_flip=vertical_flip,
    brightness=brightness,
    contrast=contrast,
    resume_from=resume_from,
    show_progress=show_progress,
)
run_dir = config.run_dir
print(f"====================\nrun_name: {run_name}\n====================\n")
print(f"device: {config.device}\nmodel: {config.model_name}\nrun_dir: {run_dir}")
config_view = config.to_dict()
for key in [
    "batch_size",
    "max_epochs",
    "image_min_size",
    "image_max_size",
    "early_stopping_patience",
]:
    print(f"{key}: {config_view[key]}")
print(f"resume_from: {config.resume_from}")

## 입력 준비 및 검증

`data_factory_path`가 가리키는 함수는 `factory(config=config, device=device)` 호출을 지원하고 `(train_loader, val_loader, label_map, data_summary)`를 반환해야 한다. 정확한 형식은 `docs/specs/detector_input_spec.md`를 따른다.

In [ ]:
training_data = prepare_detector_data(config)

In [ ]:
train_loader = training_data.train_loader
val_loader = training_data.val_loader
label_map = training_data.label_map
data_summary = training_data.data_summary
device = training_data.device
print(
    f"입력 검증 완료: device={device}, "
    f"train={data_summary['num_train_images']}, "
    f"validation={data_summary['num_val_images']}, "
    f"classes={label_map['num_foreground_classes']}"
)

## 학습 전 bbox 확인

입력 검증은 원본 annotation 자체에서 빠진 알약을 찾을 수 없다. 학습 전에 여러 무작위 샘플에서 이미지 속 모든 알약에 bbox가 있는지 확인한다.

In [ ]:
preview_batch = next(iter(val_loader))
preview_figure = plot_detection_comparison(
    preview_batch["images"][0],
    [],
    target=preview_batch["targets"][0],
    title=f"GT image_id={preview_batch['meta'][0]['image_id']}",
)
preview_figure

## 학습 또는 재개

재개할 때는 `config.resume_from`을 같은 실행의 `last.pt`로 설정한다. 첫 실행에서는 ImageNet backbone 가중치를 내려받을 수 있다.

In [ ]:
# config.resume_from = output_dir / run_name / "last.pt"
training_result = train_detector(
    config,
    data=training_data,
)
training_result

## 학습 지표 확인

epoch별 loss, average precision, recall과 예측 개수 오차를 확인한다.

In [ ]:
metrics = pd.read_csv(training_result.run_dir / "metrics.csv")
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics.plot(x="epoch", y="train_loss", ax=axes[0, 0], title="Training loss")
metrics.plot(
    x="epoch",
    y=["map", "map_50", "map_75", "map_75_95"],
    ax=axes[0, 1],
    title="Validation average precision",
)
metrics.plot(
    x="epoch",
    y=["mean_average_recall", "recall_before_threshold", "recall_after_threshold"],
    ax=axes[1, 0],
    title="Validation recall",
)
metrics.plot(
    x="epoch",
    y=["mean_count_error_before_threshold", "mean_count_error_after_threshold"],
    ax=axes[1, 1],
    title="Prediction count error",
)
fig.tight_layout()

## Best checkpoint 평가 및 artifact 저장

In [ ]:
best_model, device = load_detector_inference_model(
    training_result.best_checkpoint_path,
    label_map,
    device=device,
)
best_metrics = evaluate_model(
    best_model,
    val_loader,
    device,
    score_threshold=config.score_threshold,
    show_progress=config.show_progress,
    progress_description="Detector final validation",
)
inference_result = predict_detector(
    best_model,
    val_loader,
    device,
    score_threshold=config.score_threshold,
    detector_name=config.model_name,
)
save_inference_artifacts(inference_result, training_result.run_dir)
best_metrics.to_dict()

## 예측 시각화

입력 확인에 사용한 validation 이미지의 정답 bbox와 best checkpoint 예측을 비교해 PNG로 저장한다.

In [ ]:
first_image_id = preview_batch["meta"][0]["image_id"]
first_predictions = [
    prediction
    for prediction in inference_result.predictions
    if prediction.image_id == first_image_id
]
comparison_figure = plot_detection_comparison(
    preview_batch["images"][0],
    first_predictions,
    target=preview_batch["targets"][0],
    title=f"GT vs prediction image_id={first_image_id}",
)
visualization_dir = training_result.run_dir / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
comparison_figure.savefig(
    visualization_dir / f"image-{first_image_id}.png",
    bbox_inches="tight",
)
comparison_figure

## 선택: 분류기 연결 batch 확인

아래 셀은 crop batch만 만든다. 분류기 모델과 class map은 분류기 실행 코드에서 직접 전달한다.

In [ ]:
classifier_batch = build_classifier_batch(
    preview_batch["images"][0],
    first_predictions,
)
print(
    f"classifier images={tuple(classifier_batch['images'].shape)}, "
    f"detections={len(classifier_batch['meta'])}"
)